# Sprint 2 Role 2 — Data Validation

Re-runnable companion to `docs/data_validation.md` (issue #21). Loads the
merged `opportunity_df` from `src.opportunity_cleaner.build_opportunity_df()`,
applies Yixiao's `clean_opportunity_df` to get the Sprint 2
`cleaned_opportunity_df`, and calls each function in `src.data_validator` so
every number in the notes document can be reproduced from a clean kernel.

Run order: setup → missingness → revenue → date/duration → categorical →
data-quality flags → status_reason taxonomy → owner identity → owner
aggregates → field reliability for scoring.

## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

from src.opportunity_cleaner import build_opportunity_df, clean_opportunity_df
from src.data_validator import (
    Fields,
    apply_revenue_hierarchy,
    build_field_reliability_report,
    build_owner_aggregates,
    classify_opportunity_outcome,
    summarize_missingness,
    validate_categorical_fields,
    validate_date_duration_fields,
    validate_owner_identity,
    validate_quality_flags,
    validate_revenue_fields,
)

opportunity_df, schema_comparison, merge_summary, audit_tables = build_opportunity_df()

# Sprint 2 input: the Week 1 merge plus Yixiao's additive cleaning pass
# (merge-provenance flags, data-quality flags, precomputed authoritative_revenue).
# Every validator below runs on this cleaned frame.
cleaned_opportunity_df = clean_opportunity_df(opportunity_df)
print(
    f"opportunity_df:         {len(opportunity_df):,} rows, "
    f"{opportunity_df.shape[1]} columns"
)
print(
    f"cleaned_opportunity_df: {len(cleaned_opportunity_df):,} rows, "
    f"{cleaned_opportunity_df.shape[1]} columns"
)

## 1. Missingness summary

One row per validated business field. The columns most central to scoring
(`status`, `sales_stage`, `opportunity_owner`, `opportunity_manager`) are
100% populated. The 12.2% null on `opportunity_estimated_revenue_base_cad`
is the unmatched opps2 rows, not a defect (see §2).

In [ ]:
missingness = summarize_missingness(cleaned_opportunity_df)
missingness

## 2. Revenue field hierarchy

`total_estimated_revenue` is the canonical revenue field; the fallback to
`opportunity_estimated_revenue_base_cad` covers the 51 opps1-exclusive rows
where the primary is null. As of Yixiao's Sprint 2 merge the fallback field
is also carried on matched rows, so the two now overlap on ~7,628 rows and
agree within 5% on ~99.5% of them — a cross-validation that was not possible
in Sprint 1. Service-solution revenue must **not** be summed onto either.

In [ ]:
validate_revenue_fields(cleaned_opportunity_df)

In [ ]:
# Apply the hierarchy and inspect what gets routed to fallback.
# cleaned_opportunity_df already carries a precomputed authoritative_revenue;
# apply_revenue_hierarchy recomputes it from source so the validator stays
# independent of the cleaner's version.
resolved = apply_revenue_hierarchy(cleaned_opportunity_df)
print("revenue_source value counts:")
print(resolved["revenue_source"].value_counts(dropna=False))
print("\nMatches the cleaner's precomputed column on every row:",
      bool((resolved["authoritative_revenue"].fillna(-1)
            == cleaned_opportunity_df["authoritative_revenue"].fillna(-1)).all()))
print("\nFallback rows (showing 5):")
(
    cleaned_opportunity_df.assign(**resolved)
    .loc[resolved["revenue_source"] == "fallback",
         [Fields.OWNER, Fields.STATUS, "authoritative_revenue", "revenue_source"]]
    .head()
)

## 3. Date and duration validation

Critical findings:
1. Delivery window is 100% resolvable on the Won subset (5,275 via
   `revenue_start_date`, 15 via the `close_date` fallback).
2. `n_close_before_created` is **379** on a calendar-date basis — down from
   the Sprint 1 figure of 3,049, which was a raw-timestamp artefact.
3. Duration outliers > 60 months persist (152 rows); the `clip` in
   `capacity_engine` does not address them.

In [ ]:
validate_date_duration_fields(cleaned_opportunity_df)

In [ ]:
# Investigate close_before_created on a calendar-date basis (the corrected
# comparison). created_on carries a timestamp; close_date is midnight, so
# both must be normalized before comparing — the raw-timestamp version
# inflated this count to 3,049.
created = pd.to_datetime(cleaned_opportunity_df[Fields.CREATED_ON], errors="coerce").dt.normalize()
close = pd.to_datetime(cleaned_opportunity_df[Fields.CLOSE_DATE], errors="coerce").dt.normalize()
delta_days = (close - created).dt.days
anomalies = cleaned_opportunity_df.assign(_delta_days=delta_days).loc[delta_days < 0]
print(f"Rows with close_date < created_on (calendar date): {len(anomalies):,}")
raw_delta = (
    pd.to_datetime(cleaned_opportunity_df[Fields.CLOSE_DATE], errors="coerce")
    - pd.to_datetime(cleaned_opportunity_df[Fields.CREATED_ON], errors="coerce")
).dt.days
print(f"For comparison, raw-timestamp count (the Sprint 1 artefact): {(raw_delta < 0).sum():,}")
print("\nBy status:")
print(anomalies[Fields.STATUS].value_counts())
print("\nDelta distribution (days):")
print(delta_days.loc[delta_days < 0].describe().round(1))

In [ ]:
# Duration outliers
duration = pd.to_numeric(cleaned_opportunity_df[Fields.DURATION_MONTHS], errors="coerce")
print("Duration distribution (months):")
print(duration.describe().round(1))
print("\nOver 60 months:")
print(duration[duration > 60].describe().round(1))

## 4. Categorical / scoring fields

Confirms `sales_stage` covers exactly the architecture's 7-stage table
(no unmapped values), `probability` stays in [0, 100], and the
`status` vs `status_reason` Won-detection disagreement is a 15-row corner case.

In [ ]:
validate_categorical_fields(cleaned_opportunity_df)

In [ ]:
# status x status_reason crosstab — justifies the architecture's 'use status_reason' rule
pd.crosstab(
    cleaned_opportunity_df[Fields.STATUS],
    cleaned_opportunity_df[Fields.STATUS_REASON],
    margins=True,
    margins_name="total",
)

In [ ]:
# sales_stage distribution (used for late_stage_deal_count and stage_weight mapping)
cleaned_opportunity_df[Fields.SALES_STAGE].value_counts(dropna=False)

In [ ]:
# probability distribution and null rate by status
probability = pd.to_numeric(cleaned_opportunity_df[Fields.PROBABILITY], errors="coerce")
print("Overall null rate:", probability.isna().mean().round(4))
print("\nNull rate by status:")
print(
    cleaned_opportunity_df.assign(_prob_null=probability.isna())
    .groupby(Fields.STATUS)["_prob_null"]
    .mean()
    .round(4)
)
print("\nValue distribution:")
print(probability.describe().round(2))

## 4. Data-quality and merge flags

`cleaned_opportunity_df` carries four merge-provenance flags and eight
data-quality flags. `duplicate_flag` is 0 (the merge is clean on
`opportunity_id`); `unmatched_flag` is 1,067 legitimate opps2-only rows.
The two `flag_discrepancy_*` rows reconcile the flags against the field
validators.

In [ ]:
# validate_quality_flags reads the cleaned frame's merge + data-quality flags.
# The two flag_discrepancy_* rows are cross-checks against the field
# validators: missing_revenue reconciles to 0; close_before_created shows the
# cleaner's flag (3,049) still uses the un-normalized comparison vs the
# corrected count (379).
validate_quality_flags(cleaned_opportunity_df)

## 5. status_reason outcome taxonomy

`status_reason` carries `Cancelled ...` and `Duplicated` values beyond
Won / Open / Lost. Sprint 2 decision is **Option B**: Cancelled is lost-like
(a real pipeline exit), Duplicated is carved into its own bucket and excluded
from win/loss counts. The profiling cell below is the evidence — the key fact
is that `status_reason == "Duplicated"` has **zero** overlap with the
merge-level `duplicate_flag` and is concentrated in just 3 owners, so it is a
CRM data-hygiene label, not a sales outcome. `classify_opportunity_outcome`
is the shared helper that implements this; see `docs/data_validation.md` §5.

In [ ]:
# status_reason profiling — the evidence behind the Option B decision.
print("status value counts:")
print(cleaned_opportunity_df[Fields.STATUS].value_counts(dropna=False))
print("\nstatus_reason value counts:")
print(cleaned_opportunity_df[Fields.STATUS_REASON].value_counts(dropna=False))

# The key question: does status_reason == "Duplicated" coincide with the
# merge-level duplicate_flag? If it did, the dedup in build_owner_aggregates
# would already handle it. It does not.
sr = cleaned_opportunity_df[Fields.STATUS_REASON].astype("string").str.strip().str.lower()
is_duplicated = sr.eq("duplicated")
print("\nstatus_reason == 'Duplicated' vs duplicate_flag:")
print(pd.crosstab(is_duplicated, cleaned_opportunity_df["duplicate_flag"], dropna=False))
print(f"\nDuplicated rows: {int(is_duplicated.sum())}, "
      f"owners affected: {cleaned_opportunity_df.loc[is_duplicated, Fields.OWNER].nunique()}")

In [ ]:
# classify_opportunity_outcome — the shared Option B taxonomy helper.
outcome = classify_opportunity_outcome(cleaned_opportunity_df)
print("Outcome bucket counts:")
print(outcome.value_counts(dropna=False))
print("\nReconciles to row count:", int(outcome.notna().sum()) == len(cleaned_opportunity_df))
print("\nThe 15 status==Won / null-status_reason rows now resolve to 'won':")
null_reason = cleaned_opportunity_df[Fields.STATUS_REASON].isna()
print(outcome[null_reason].value_counts())

## 6. Owner identity

24 distinct owners, no casing variants, zero owner-equals-manager rows.
Seven owners have fewer than 10 lifetime opportunities — their baselines
should carry a Low reliability flag.

In [ ]:
validate_owner_identity(cleaned_opportunity_df)

In [ ]:
# Owner deal counts (sorted, top + tail)
owner_counts = cleaned_opportunity_df[Fields.OWNER].value_counts()
print("Top 5 by lifetime opportunity count:")
print(owner_counts.head())
print("\nBottom 7 (Low reliability candidates):")
print(owner_counts.tail(7))

## 7. Owner aggregates

`build_owner_aggregates(cleaned_opportunity_df)` produces the ten columns
Freya's dashboard mock expects but Lyken's `director_df` does not yet emit —
including the Sprint 2 outcome counts (`open_`, `lost_`,
`duplicate_opportunity_count`) derived from `classify_opportunity_outcome`.
Duplicate join-key rows are deduplicated on `opportunity_id` before
aggregating. Lyken's score columns join onto this on `opportunity_owner`.

In [ ]:
owner_aggregates = build_owner_aggregates(cleaned_opportunity_df)
print(f"{len(owner_aggregates)} owners")
print("baseline_reliability distribution:")
print(owner_aggregates["baseline_reliability"].value_counts())
owner_aggregates.sort_values("weighted_pipeline_revenue", ascending=False)

In [ ]:
# Optional local export — overwrites previous run; data/processed/ is gitignored
from pathlib import Path
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
owner_aggregates.to_csv(out_dir / "owner_validation_summary.csv", index=False)
print("Wrote", out_dir / "owner_validation_summary.csv")

## 8. Field reliability for scoring

`build_field_reliability_report` rates each validated field as a scoring
input — `reliable` / `use_with_care` / `not_yet` — from computed null and
anomaly rates, plus a `usable_for_scoring` flag and the governing
`fallback_assumptions.yaml` rule. This replaces the Sprint 1 hand-rolled
colour-code cell. The human-readable version is `docs/data_validation.md` §8;
the targeted Lyken handoff is `docs/scoring_input_reliability_handoff.md`.

In [ ]:
field_reliability = build_field_reliability_report(cleaned_opportunity_df)
print("reliability distribution:", dict(field_reliability["reliability"].value_counts()))
print("usable_for_scoring:", dict(field_reliability["usable_for_scoring"].value_counts()))
field_reliability

---

**Use of Generative AI.** Anthropic Claude (Opus 4.7) was used for drafting and editing assistance on this notebook. All numerical findings come from running the validator functions on the team's `cleaned_opportunity_df`. No CGI data was sent to the tool.